In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}



def export_figure(figure, figure_id, tight=False):
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
ORDER_COLORS = {1: "#009E73", 2: "#0072B2", 3: "#E69F00", 4: "#D55E00", 5: "#CC79A7", 6: "#777777", 7: "#8A6A00"}
ORDER_STYLES = {1: (0, (6, 1)), 2: "-", 3: "--", 4: "-.", 5: (0, (5, 1, 1, 1)), 6: ":", 7: (0, (2, 1, 2, 1, 6, 1))}
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "font.size": 10, "axes.titlesize": 11.5, "axes.labelsize": 11,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5, "legend.fontsize": 8.5,
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.linewidth": 0.8,
    "axes.spines.top": True, "axes.spines.right": True,
    "figure.facecolor": "white", "axes.facecolor": "white",
})
def style_axis(axis, letter, title):
    axis.set_title(title, fontweight="normal", pad=10)
    axis.text(-0.16, 1.055, f"({letter})", transform=axis.transAxes,
              fontsize=11, fontweight="bold", ha="left", va="bottom")
    axis.tick_params(direction="out", length=3.5, width=0.8)
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
    axis.grid(alpha=0.14, which="major", linewidth=0.5)
    axis.set_xlim(0, 200)
    axis.set_xticks((0, 50, 100, 150, 200))

def circuit_panel(axis, task):
    axis.set_axis_off();axis.set_xlim(-.8,11);axis.set_ylim(0,4.65)
    axis.set_title("4-qubit circuit and target", fontweight="normal", pad=10)
    axis.text(-.095,1.055,"(a)",transform=axis.transAxes,fontsize=11,fontweight="bold",ha="left",va="bottom")
    rows=(3.55,2.65,1.75,.85)
    for qubit,y in enumerate(rows):
        axis.plot((.65,10.8),(y,y),color="black",lw=.8)
        axis.text(.45,y,rf"$q_{qubit}:|0\rangle$",ha="right",va="center",fontsize=8)
        for x,k in ((2.1,qubit),(6.05,qubit+4)):
            letter="Y" if k<4 else "Z"
            axis.add_patch(Rectangle((x-.91,y-.30),1.82,.60,facecolor="white",edgecolor="#0072B2",lw=.8,zorder=3))
            axis.text(x,y+.11,rf"$R_{letter}(x_{k})$",ha="center",va="center",fontsize=8,zorder=4)
            axis.text(x,y-.15,rf"$\mathcal{{N}}_{{{letter},\gamma}}$",ha="center",va="center",fontsize=7.7,zorder=4)
    for j,x in enumerate((3.4,4.2,5.0)):
        upper,lower=rows[j:j+2]
        axis.plot((x,x),(lower,upper),color="black",lw=.8)
        axis.plot((x,x),(lower,upper),"o",color="black",ms=2.6)
        axis.text(x,(upper+lower)/2,"CZ",fontsize=7,ha="center",va="center",bbox={"facecolor":"white","edgecolor":"none","pad":.4})
    axis.add_patch(Rectangle((8,.48),2.25,3.5,facecolor="white",edgecolor="#56B4E9",lw=.9,zorder=3))
    axis.text(9.125,2.2,r"$V(\boldsymbol{\theta})$",ha="center",va="center",fontsize=10,zorder=4)
    target=(r"$y_\gamma(\mathbf{x})=F_\gamma(\mathbf{x};\mathbf{0})$" if task=="matched" else r"$y_0(\mathbf{x})=F_0(\mathbf{x};\mathbf{0})$")
    axis.text(5.2,.10,target,ha="center",va="center",fontsize=8.6)

In [ ]:
raw=load_npz("data/Fig6/main.npz")
time=raw["time"];condition=raw["condition_index"];losses=raw["sector_loss"];seeds=raw["seeds"]
ratios=losses/losses[:1]
previous_limits=json.loads((PACKAGE_ROOT/"data/Fig6/main_limits.json").read_text())
ORDERS=tuple(range(1,8))
for ci in range(4):
    assert (condition==ci).sum()==20
assert losses.shape[-1]==9

In [ ]:
fig = plt.figure(figsize=(11.5, 8.4))
rows = fig.add_gridspec(2, 1, left=.075, right=.985, bottom=.095,
                       top=.93, hspace=.46)
top = rows[0].subgridspec(1, 2, wspace=.28)
bottom = rows[1].subgridspec(1, 3, wspace=.37)
circuit = fig.add_subplot(top[0, 0])
noiseless = fig.add_subplot(top[0, 1])
noisy = [fig.add_subplot(bottom[0, j]) for j in range(3)]
circuit_panel(circuit, "common")
panels = [(noiseless, 0, "b", "Noiseless", True, "common_c"),
          (noisy[0], 1, "c", "Encoding-axis Pauli\n" + r"$\gamma=0.05$",
           True, "gamma005_condition1"),
          (noisy[1], 2, "d", "Encoding-axis Pauli\n" + r"$\gamma=0.15$",
           False, "common_d"),
          (noisy[2], 3, "e", "Encoding-axis Pauli\n" + r"$\gamma=0.30$",
           False, "common_e")]
iterations = np.arange(len(time), dtype=int)
assert np.array_equal(time, 0.10 * iterations)
plot_data = {"time": time, "iteration": iterations}
audit = []
handles = []
for ax, ci, letter, title, logarithmic, old_prefix in panels:
    style_axis(ax, letter, title)
    ax.set_xlim(0, 2000)
    ax.set_xticks((0, 500, 1000, 1500, 2000))
    if letter == "b":
        ax.texts[0].set_x(-.095)
    ax.set_yscale("log" if logarithmic else "linear")
    ax.set_ylim(previous_limits["noiseless_log_limits"] if logarithmic
                else previous_limits["linear_sector_limits"])
    take = condition == ci
    for k in ORDERS:
        quantiles = np.quantile(ratios[:, take, k], [.25, .5, .75], axis=1)
        direct = np.quantile(losses[:, take, k] / losses[0, take, k],
                             [.25, .5, .75], axis=1)
        assert np.array_equal(quantiles, direct)
        if logarithmic:
            assert np.all(quantiles[0] > 0)
        ymin, ymax = ax.get_ylim()
        assert quantiles[0].min() >= ymin and quantiles[2].max() <= ymax
        color, linestyle = ORDER_COLORS[k], ORDER_STYLES[k]
        ax.fill_between(iterations, quantiles[0], quantiles[2], color=color,
                        alpha=.08, linewidth=0)
        line, = ax.plot(iterations, quantiles[1], color=color, linestyle=linestyle,
                        linewidth=1.1)
        assert np.array_equal(line.get_ydata(), quantiles[1])
        assert np.array_equal(line.get_xdata(), iterations)
        if ci == 0:
            handles.append(line)
        key = f"panel_{letter}_gamma_{(0., .05, .15, .30)[ci]}_k{k}"
        plot_data[key] = quantiles
    ax.axhline(1, color=".4", ls="--", lw=.65, zorder=0)
    ax.axhline(.5, color=".55", ls=":", lw=.65, zorder=0)
noiseless.set_ylabel(r"Relative sector loss $R_k(t)$")
noisy[0].set_ylabel(r"Relative sector loss $R_k(t)$")
idx = (0, 4, 1, 5, 2, 6, 3)
fig.legend([handles[j] for j in idx], [rf"$k={ORDERS[j]}$" for j in idx],
           loc="center", bbox_to_anchor=(.53, .515), ncol=4, frameon=False,
           handlelength=2.5, columnspacing=1.1)
fig.supxlabel("Training iteration", fontsize=11, y=.015)

export_figure(fig,"Fig6",tight=True)
plt.close(fig)